In [1]:
import numpy as np

N = 2048

A = np.random.rand(N,N).astype(np.float32)

data = A.flatten()

In [2]:
import numpy as np
from numba import cuda,float32

N = 2048

A = np.random.rand(N,N).astype(np.float32)

data = A.flatten()

THREADS = 256

@cuda.jit
def reduction_kernel(inp,out):

    shared = cuda.shared.array(256,dtype=float32)

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    stride = cuda.blockDim.x * cuda.gridDim.x

    temp = 0

    # Loop Unrolling
    while idx < inp.size:

        temp += inp[idx]

        if idx + cuda.blockDim.x < inp.size:
            temp += inp[idx + cuda.blockDim.x]

        idx += stride*2

    shared[tid] = temp

    cuda.syncthreads()

    # Shared Memory Reduction

    s = cuda.blockDim.x // 2

    while s > 32:

        if tid < s:
            shared[tid] += shared[tid+s]

        cuda.syncthreads()

        s//=2

    # Warp Reduction

    if tid < 32:

        shared[tid] += shared[tid+32]
        shared[tid] += shared[tid+16]
        shared[tid] += shared[tid+8]
        shared[tid] += shared[tid+4]
        shared[tid] += shared[tid+2]
        shared[tid] += shared[tid+1]

    if tid==0:
        out[cuda.blockIdx.x]=shared[0]

# Number of blocks
blocks = 256

partial = np.zeros(blocks,dtype=np.float32)

reduction_kernel[blocks,THREADS](data,partial)

cuda.synchronize()

gpu_sum = partial.sum()

cpu_sum = data.sum()

print("GPU Sum =",gpu_sum)
print("CPU Sum =",cpu_sum)
print("Difference =",abs(cpu_sum-gpu_sum))

GPU Sum = 2097670.0
CPU Sum = 2097531.0
Difference = 139.0


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
